# Data Preprocessing

## Import Libraries

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
import scipy.stats

## Read the Data

In [4]:
client_train = pd.read_csv('../data/client_train.csv', low_memory=False)
invoice_train = pd.read_csv('../data/invoice_train.csv', low_memory=False)

client_test = pd.read_csv(f'../data/client_test.csv', low_memory=False)
invoice_test = pd.read_csv(f'../data/invoice_test.csv', low_memory=False)
sample_submission = pd.read_csv(f'../results/SampleSubmission.csv', low_memory=False)

In [ ]:
client_train.head()

,disrict,client_id,client_catg,region,creation_date,target
0,60,train_Client_0,11,101,31/12/1994,0.0
1,69,train_Client_1,11,107,29/05/2002,0.0
2,62,train_Client_10,11,301,13/03/1986,0.0
3,69,train_Client_100,11,105,11/07/1996,0.0
4,62,train_Client_1000,11,303,14/10/2014,0.0


## Data Cleaning

In [ ]:
train = invoice_train.copy()
# counter_statue == 269375 -> can be droped, since there is just one observation and also the train_Client_53725 belong to only that observation 
train = train.query("counter_statue != 269375")
# counter_statue == 420 and reading_remarque == 5 and counter_code == 1 -> they are just one time in the same observation, so we can drop them as well
train = train.query("counter_statue != 420")
# replace string of int to int
train['counter_statue'] = train['counter_statue'].replace('0', 0)
train['counter_statue'] = train['counter_statue'].replace('1', 1)
train['counter_statue'] = train['counter_statue'].replace('4', 4)
train['counter_statue'] = train['counter_statue'].replace('5', 5)
# counter_statue == 769 and reading_remarque == 207 ???
train = train.query("counter_statue != 769")
# counter_statue == 618 and reading_remarque == 413 ???
train = train.query("counter_statue != 618")
# counter_statue == 46 and reading_remarque == 203 ???
train = train.query("counter_statue != 46")
#months_number > 88
train = train.query("months_number <= 88") # auch aus testdaten
invoice_train = train.copy()

invoice_test = invoice_test.query("months_number <= 88")

## Train Test Split

In [ ]:
# split client train, stratify by target variable, and set random state for reproducibility
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(client_train.drop(columns=['target']), client_train['target'], test_size=0.1, stratify=client_train['target'], random_state=42)

In [ ]:
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)

(121943, 5) (121943,)
(13550, 5) (13550,)


In [ ]:
client_train = pd.merge(X_train, y_train, left_index=True, right_index=True)
client_val = pd.merge(X_val, y_val, left_index=True, right_index=True)

### undersampling

In [ ]:
# undersample the majority class in the training data
ros = RandomOverSampler(random_state=42, sampling_strategy=0.1)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train)
rus = RandomUnderSampler(random_state=42, sampling_strategy=0.5)
X_train_res, y_train_res = rus.fit_resample(X_train, y_train)

In [ ]:
clients_in_resampled_train = X_train_res['client_id']
clients_in_val = X_val['client_id']

invoice_train_res = invoice_train[invoice_train['client_id'].isin(clients_in_resampled_train)]
invoice_val = invoice_train[invoice_train['client_id'].isin(clients_in_val)]

client_train_res = pd.merge(X_train_res, y_train_res, left_index=True, right_index=True)

In [ ]:
print(X_train_res.shape, y_train_res.shape)
print(X_val.shape, y_val.shape)

(20427, 5) (20427,)
(13550, 5) (13550,)


## Feature Engineering Functions

In [ ]:
#convert the column invoice_date to date time format on both the invoice train and invoice test
def invoice_convert_date(df):
    df_updated = df.copy()
    df_updated['invoice_date'] = pd.to_datetime(df_updated['invoice_date'])
    return df_updated

In [ ]:
# convert tarif_type, counter_statue, counter_code, reading_remarque, counter_type to categorical data type
def invoice_to_category(df):
    df_updated = df.copy()
    df_updated['tarif_type'] = df_updated['tarif_type'].astype('category')
    df_updated['counter_statue'] = df_updated['counter_statue'].astype('category')
    df_updated['counter_code'] = df_updated['counter_code'].astype('category')
    df_updated['reading_remarque'] = df_updated['reading_remarque'].astype('category')
    df_updated['counter_type'] = df_updated['counter_type'].astype('category')
    return df_updated

In [ ]:
# create variable that sums the consumption for all 4 levels
def invoice_create_consumption(df):
    df_updated = df.copy()
    df_updated['total_consumption'] = df_updated['new_index'] - df_updated['old_index']
    df_updated['consumption_per_month'] = df_updated['total_consumption'] / df_updated['months_number']
    return df_updated

In [ ]:
# convert disrict, client_catg, region to categorical data type
def client_to_category(df):
    df_updated = df.copy()
    df_updated['disrict'] = df_updated['disrict'].astype('category')
    df_updated['client_catg'] = df_updated['client_catg'].astype('category')
    df_updated['region'] = df_updated['region'].astype('category')
    return df_updated

In [ ]:
# convert creation_date to date time format on both the client train and client test
def client_convert_date(df):
    df_updated = df.copy()
    df_updated['creation_date'] = pd.to_datetime(df_updated['creation_date'], dayfirst=True)
    # create new variable with 7 bins for the creation date using cut
    df_updated['creation_date_bin'] = pd.cut(df_updated['creation_date'], bins=7, labels=False)
    # convert the creation_date_bin to categorical data type
    df_updated['creation_date_bin'] = df_updated['creation_date_bin'].astype('category')
    return df_updated

In [ ]:
# DEFINE LEVEL USAGE
# -------------------------
def define_activity_level4_usage(df):
    df_updated = df.copy()
    df_updated['used_lvl4'] = (df_updated['consommation_level_4'] > 0).astype(int)
    # activity
    # any consumption at all (levels 1–4)
    df_updated['active'] = (
        (df_updated['consommation_level_1'] > 0) |
        (df_updated['consommation_level_2'] > 0) |
        (df_updated['consommation_level_3'] > 0) |
        (df_updated['consommation_level_4'] > 0)
    ).astype(int)
    return df_updated

In [ ]:
# -------------------------
# FRACTION PER CLIENT
# -------------------------
def lvl4_fraction(df):
    df_updated = df.copy()
    client_lvl4_fraction = df_updated.groupby('client_id')['used_lvl4'].mean().reset_index()
    client_lvl4_fraction.rename(columns={
        'used_lvl4': 'frac_time_lvl4'
        }, inplace=True)
    df_updated['frac_time_lvl4'] = df_updated['client_id'].map(
        client_lvl4_fraction.set_index('client_id')['frac_time_lvl4']
    )
    return df_updated

In [ ]:
def duplicate_invoices_same_day(df):
    df = df.copy()
    grp = df.groupby(['client_id', 'counter_type', 'invoice_date'])['client_id']
    # count rows per (client_id, counter_type, invoice_date)
    per_day_count = grp.transform('size')
    # per-row boolean: this row belongs to a duplicated same-day group
    duplicated_group = per_day_count.gt(1)
    # per-client flag: any duplicated same-day group for that client
    df['has_multi_invoice_same_day'] = duplicated_group.groupby(df['client_id']).transform('any').astype('int8')
    return df

In [ ]:
# date gaps
def date_gaps(df):
    df_updated = df.copy()
    df_updated = df_updated.sort_values(['client_id', 'invoice_date'])
    df_updated['gap_days'] = df_updated.groupby(['client_id', 'counter_type'])['invoice_date'].diff().dt.days
    return df_updated

In [ ]:
def invoice_scaled_consumption_per_month(df):
    df_updated = df.copy()
    df_updated['consumption_per_month_scaled'] = df_updated.groupby(['client_id', 'counter_type'])['consumption_per_month'].transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0.5)
    return df_updated

In [ ]:
def correct_newIndex(df):
    updated_df = df.copy()
    updated_df['index_diff'] = updated_df['new_index'] - updated_df['old_index']
    updated_df.loc[updated_df['index_diff'] < 0, 'new_index'] += 100000
    return updated_df

### Run on (undersampled) data

In [ ]:
invoice_train_res = invoice_convert_date(invoice_train_res)
invoice_train_res = correct_newIndex(invoice_train_res)
invoice_train_res = invoice_to_category(invoice_train_res)
invoice_train_res = invoice_create_consumption(invoice_train_res)
invoice_train_res = define_activity_level4_usage(invoice_train_res)
invoice_train_res = lvl4_fraction(invoice_train_res)
invoice_train_res = duplicate_invoices_same_day(invoice_train_res)
invoice_train_res = date_gaps(invoice_train_res)
invoice_train_res = invoice_scaled_consumption_per_month(invoice_train_res)
# write the preprocessed invoice train_res to csv
invoice_train_res.to_csv('data/created_invoice_train_res.csv', index=False)

In [ ]:
invoice_val = invoice_convert_date(invoice_val)
invoice_val = correct_newIndex(invoice_val)
invoice_val = invoice_to_category(invoice_val)
invoice_val = invoice_create_consumption(invoice_val)
invoice_val = define_activity_level4_usage(invoice_val)
invoice_val = lvl4_fraction(invoice_val)
invoice_val = duplicate_invoices_same_day(invoice_val)
invoice_val = date_gaps(invoice_val)
invoice_val = invoice_scaled_consumption_per_month(invoice_val)
# write the preprocessed invoice val to csv
invoice_val.to_csv('data/created_invoice_val.csv', index=False)

In [ ]:
invoice_test = correct_newIndex(invoice_test)
invoice_test = invoice_convert_date(invoice_test)
invoice_test = invoice_to_category(invoice_test)
invoice_test = invoice_create_consumption(invoice_test)
invoice_test = define_activity_level4_usage(invoice_test)
invoice_test = lvl4_fraction(invoice_test)
invoice_test = duplicate_invoices_same_day(invoice_test)
invoice_test = date_gaps(invoice_test)
invoice_test = invoice_scaled_consumption_per_month(invoice_test)
# write the preprocessed invoice test to csv
invoice_test.to_csv('data/created_invoice_test.csv', index=False)

In [ ]:
for df in [client_train_res, client_val, client_test]:
    df = client_convert_date(df)
    df = client_to_category(df)

# client_train_res.to_csv('data/created_client_train_res.csv', index=False)
# client_val.to_csv('data/created_client_val.csv', index=False)
# client_test.to_csv('data/created_client_test.csv', index=False)

In [ ]:
# read created train_res data
# invoice_train_res = pd.read_csv('data/created_invoice_train_res.csv')
# read created data
# invoice_val = pd.read_csv('data/created_invoice_val.csv')
# read created data
# invoice_test = pd.read_csv('data/created_invoice_test.csv')

# read created client data
# client_train_res = pd.read_csv('data/created_client_train_res.csv')
# client_val = pd.read_csv('data/created_client_val.csv')
# client_test = pd.read_csv('data/created_client_test.csv')

In [ ]:
invoice_train_res.tail()

,client_id,invoice_date,tarif_type,counter_number,counter_statue,counter_code,reading_remarque,counter_coefficient,consommation_level_1,consommation_level_2,...,counter_type,index_diff,total_consumption,consumption_per_month,used_lvl4,active,frac_time_lvl4,has_multi_invoice_same_day,gap_days,consumption_per_month_scaled
4476497,train_Client_99990,2017-09-19,11,416794,0,207,9,1,346,0,...,ELEC,346,346,86.50,0,1,0.0,0,124.0,0.222741
4476496,train_Client_99990,2018-01-24,11,416794,0,207,9,1,416,0,...,ELEC,416,416,104.00,0,1,0.0,0,127.0,0.331776
4476495,train_Client_99990,2018-05-17,11,416794,0,207,9,1,337,0,...,ELEC,337,337,84.25,0,1,0.0,0,113.0,0.208723
4476502,train_Client_99990,2019-01-25,11,416794,0,207,8,1,110,0,...,ELEC,110,110,55.00,0,1,0.0,0,253.0,0.026480
4476503,train_Client_99990,2019-05-22,11,416794,0,207,8,1,800,45,...,ELEC,845,845,211.25,0,1,0.0,0,117.0,1.000000


## Aggregate functions on invoice to client

In [ ]:
def aggregate_by_client_id(invoice_data):
    aggs = {}
    aggs['gap_days'] = [np.nanmean]
    aggs['gap_days'] = [np.nanstd]
    aggs['consumption_per_month'] = [np.nanmean]
    aggs['consumption_per_month'] = [np.nanstd]
    aggs['frac_time_lvl4'] = [np.nanmax]
    aggs['consumption_per_month_scaled'] = [np.nanstd]
    aggs['active'] = ['sum']
    aggs['active'] = [np.nanmean]
    aggs['counter_statue'] = [np.nanstd]
    aggs['reading_remarque'] = [np.nanstd]
    aggs['tarif_type'] = [scipy.stats.mode]
    aggs['index_diff'] = [np.nanstd]
    aggs['has_multi_invoice_same_day'] = [np.nanmax]
    aggs['used_lvl4'] = ['sum']


    agg_trans = invoice_data.groupby(['client_id']).agg(aggs)
    agg_trans.columns = ['_'.join(col).strip() for col in agg_trans.columns.values]
    agg_trans.reset_index(inplace=True)

    df = (invoice_data.groupby('client_id')
            .size()
            .reset_index(name='{}transactions_count'.format('1')))
    return pd.merge(df, agg_trans, on='client_id', how='left')

In [ ]:
#group invoice data by client_id
agg_train_res = aggregate_by_client_id(invoice_train_res)
agg_val = aggregate_by_client_id(invoice_val)
agg_test = aggregate_by_client_id(invoice_test)

In [ ]:
agg_train_res.head()

,client_id,1transactions_count,gap_days_nanstd,consumption_per_month_nanstd,frac_time_lvl4_nanmax,consumption_per_month_scaled_nanstd,active_nanmean,counter_statue_nanstd,reading_remarque_nanstd,tarif_type_mode,index_diff_nanstd,has_multi_invoice_same_day_nanmax,used_lvl4_sum
0,train_Client_100000,40,46.310659,89.272755,0.0,0.264030,0.925000,0.405096,1.165476,"(11, 20)",338.205437,0,0
1,train_Client_100016,79,102.775580,54.589136,0.0,0.238825,0.962025,0.000000,1.357657,"(11, 40)",218.769506,0,0
2,train_Client_100019,33,88.170198,319.802735,0.0,0.259527,1.000000,0.000000,1.118881,"(11, 33)",1394.529842,0,0
3,train_Client_100029,6,116.218759,80.570841,0.0,0.402351,0.500000,0.000000,0.516398,"(11, 3)",322.283364,1,0
4,train_Client_100030,1,NaN,NaN,0.0,NaN,1.000000,NaN,NaN,"(11, 1)",NaN,0,0


In [ ]:
# replace tarif_type_mode by first entry of that column
agg_train_res['tarif_type_mode'] = agg_train_res['tarif_type_mode'].apply(lambda x: x[0] if isinstance(x, tuple) else x)
# change to category type
agg_train_res['tarif_type_mode'] = agg_train_res['tarif_type_mode'].astype('category')

# same for val and test
agg_val['tarif_type_mode'] = agg_val['tarif_type_mode'].apply(lambda x: x[0] if isinstance(x, tuple) else x)
agg_val['tarif_type_mode'] = agg_val['tarif_type_mode'].astype('category')
agg_test['tarif_type_mode'] = agg_test['tarif_type_mode'].apply(lambda x: x[0] if isinstance(x, tuple) else x)
agg_test['tarif_type_mode'] = agg_test['tarif_type_mode'].astype('category')

In [ ]:
#merge aggregate data with client dataset
train_res = pd.merge(client_train_res, agg_train_res, on='client_id', how='left')
display(train_res)

train_res.to_csv('data/created_train_res.csv', index=False)

#same for val 
val = pd.merge(client_val, agg_val, on='client_id', how='left')
display(val)

val.to_csv('data/created_val.csv', index=False)

# merge client test with agg test
test = pd.merge(client_test, agg_test, on='client_id', how='left')
display(test)

test.to_csv('data/created_test.csv', index=False)

,disrict,client_id,client_catg,region,creation_date,target,1transactions_count,gap_days_nanstd,consumption_per_month_nanstd,frac_time_lvl4_nanmax,consumption_per_month_scaled_nanstd,active_nanmean,counter_statue_nanstd,reading_remarque_nanstd,tarif_type_mode,index_diff_nanstd,has_multi_invoice_same_day_nanmax,used_lvl4_sum
0,62,train_Client_37283,11,310,27/08/2016,0.0,8.0,63.537016,124.597114,0.000000,0.415497,1.000000,0.000000,0.000000,11,818.388737,0.0,0.0
1,69,train_Client_54534,11,107,05/06/2010,0.0,54.0,76.736842,64.658691,0.000000,0.244157,0.962963,0.190626,1.345853,11,259.792042,0.0,0.0
2,63,train_Client_2282,11,311,21/12/1986,0.0,130.0,80.471860,563.704473,0.000000,0.278937,0.530769,0.000000,0.315094,11,8734.396193,0.0,0.0
3,69,train_Client_44909,11,104,01/12/2009,0.0,11.0,229.327277,86.042191,0.000000,0.363982,1.000000,0.000000,1.361817,40,789.985224,0.0,0.0
4,63,train_Client_98470,11,101,03/03/1994,0.0,83.0,102.970386,220.296235,0.000000,0.413606,0.927711,0.000000,1.295577,40,881.184938,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20422,60,train_Client_98914,11,101,18/02/1986,1.0,70.0,93.620932,193.935589,0.014286,0.261838,1.000000,0.000000,1.252823,11,779.389893,0.0,1.0
20423,69,train_Client_94021,11,107,13/09/2000,1.0,35.0,77.832582,248.761954,0.028571,0.256588,0.971429,0.000000,1.235334,11,1127.753497,0.0,1.0
20424,62,train_Client_76339,11,302,20/11/1984,1.0,34.0,108.056645,58.621138,0.000000,0.241488,1.000000,0.000000,1.041047,11,305.974672,0.0,0.0
20425,60,train_Client_65563,11,101,20/02/2009,1.0,37.0,206.453092,33.241921,0.000000,0.236067,0.972973,0.164399,1.150793,40,254.426190,0.0,0.0


,disrict,client_id,client_catg,region,creation_date,target,1transactions_count,gap_days_nanstd,consumption_per_month_nanstd,frac_time_lvl4_nanmax,consumption_per_month_scaled_nanstd,active_nanmean,counter_statue_nanstd,reading_remarque_nanstd,tarif_type_mode,index_diff_nanstd,has_multi_invoice_same_day_nanmax,used_lvl4_sum
0,62,train_Client_124396,11,309,08/07/2010,0.0,23.0,81.958177,24.251929,0.000000,0.282821,0.391304,0.208514,1.206045,11,154.179840,0.0,0.0
1,60,train_Client_53991,11,101,06/06/1989,0.0,31.0,100.309985,64.004147,0.000000,0.289612,1.000000,0.000000,1.222548,10,270.678494,0.0,0.0
2,62,train_Client_106081,11,303,21/05/2002,0.0,79.0,119.516999,68.579467,0.000000,0.269746,0.987342,0.000000,1.407888,11,313.416317,0.0,0.0
3,62,train_Client_22679,11,310,25/12/2008,0.0,16.0,147.256563,10.416621,0.000000,0.289351,0.937500,0.000000,1.327592,11,45.944895,0.0,0.0
4,62,train_Client_65828,11,303,26/09/2000,0.0,44.0,99.224358,81.875634,0.000000,0.268583,0.931818,0.785707,1.262569,11,392.249837,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13545,63,train_Client_127166,11,306,22/06/2011,1.0,14.0,162.416882,30.368173,0.000000,0.311735,1.000000,0.000000,1.266647,11,358.559511,0.0,0.0
13546,62,train_Client_105991,11,301,21/08/1989,0.0,75.0,80.339197,94.042105,0.000000,0.233679,0.986667,0.115470,1.435960,10,416.131403,0.0,0.0
13547,69,train_Client_109115,51,103,14/04/2009,1.0,135.0,90.018698,407.250064,0.281481,0.210220,0.962963,0.000000,1.269274,11,1109.329157,0.0,38.0
13548,69,train_Client_58137,11,103,11/12/2014,0.0,26.0,82.350611,268.648956,0.038462,0.269220,0.846154,0.000000,1.104536,11,1074.173429,0.0,1.0


,disrict,client_id,client_catg,region,creation_date,1transactions_count,gap_days_nanstd,consumption_per_month_nanstd,frac_time_lvl4_nanmax,consumption_per_month_scaled_nanstd,active_nanmean,counter_statue_nanstd,reading_remarque_nanstd,tarif_type_mode,index_diff_nanstd,has_multi_invoice_same_day_nanmax,used_lvl4_sum
0,62,test_Client_0,11,307,28/05/2002,37.0,44.900834,40.424028,0.000000,0.175757,0.972973,0.000000,1.221061,11,235.684859,0.0,0.0
1,69,test_Client_1,11,103,06/08/2009,22.0,117.157647,722.359649,0.045455,0.194222,1.000000,0.213201,1.216766,11,3200.849986,0.0,1.0
2,62,test_Client_10,11,310,07/04/2004,74.0,13.134901,105.694968,0.013514,0.241883,0.986486,0.000000,1.482216,11,422.779872,0.0,1.0
3,60,test_Client_100,11,101,08/10/1992,40.0,96.299793,66.060236,0.000000,0.293850,0.950000,0.000000,1.034966,11,247.253171,0.0,0.0
4,62,test_Client_1000,11,301,21/07/1977,53.0,113.376261,158.358003,0.000000,0.280807,0.924528,0.686803,1.319443,11,864.294814,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58064,63,test_Client_9995,11,399,17/03/2010,4.0,0.000000,133.188991,0.000000,0.408248,0.500000,0.000000,0.000000,11,532.755964,0.0,0.0
58065,63,test_Client_9996,11,311,28/05/2011,46.0,58.724011,23.653269,0.000000,0.220683,0.978261,0.759163,1.100066,11,107.556838,0.0,0.0
58066,60,test_Client_9997,11,101,04/03/1978,59.0,120.360325,31.634983,0.000000,0.220892,1.000000,0.000000,1.065094,10,116.987086,0.0,0.0
58067,60,test_Client_9998,11,101,23/02/2018,1.0,NaN,NaN,0.000000,NaN,1.000000,NaN,NaN,11,NaN,0.0,0.0


In [ ]:
train_res = pd.merge(client_train_res, agg_train_res, on='client_id', how='left')
display(train_res)

train_res.to_csv('data/created_train_res.csv', index=False)

#same for val 
val = pd.merge(client_val, agg_val, on='client_id', how='left')
display(val)

val.to_csv('data/created_val.csv', index=False)

# merge client test with agg test
test = pd.merge(client_test, agg_test, on='client_id', how='left')
display(test)

test.to_csv('data/created_test.csv', index=False)

,disrict,client_id,client_catg,region,creation_date,target,1transactions_count,gap_days_nanstd,consumption_per_month_nanstd,frac_time_lvl4_nanmax,consumption_per_month_scaled_nanstd,active_nanmean,counter_statue_nanstd,reading_remarque_nanstd,tarif_type_mode,index_diff_nanstd,has_multi_invoice_same_day_nanmax,used_lvl4_sum
0,62,train_Client_37283,11,310,27/08/2016,0.0,8.0,63.537016,124.597114,0.000000,0.415497,1.000000,0.000000,0.000000,11,818.388737,0.0,0.0
1,69,train_Client_54534,11,107,05/06/2010,0.0,54.0,76.736842,64.658691,0.000000,0.244157,0.962963,0.190626,1.345853,11,259.792042,0.0,0.0
2,63,train_Client_2282,11,311,21/12/1986,0.0,130.0,80.471860,563.704473,0.000000,0.278937,0.530769,0.000000,0.315094,11,8734.396193,0.0,0.0
3,69,train_Client_44909,11,104,01/12/2009,0.0,11.0,229.327277,86.042191,0.000000,0.363982,1.000000,0.000000,1.361817,40,789.985224,0.0,0.0
4,63,train_Client_98470,11,101,03/03/1994,0.0,83.0,102.970386,220.296235,0.000000,0.413606,0.927711,0.000000,1.295577,40,881.184938,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20422,60,train_Client_98914,11,101,18/02/1986,1.0,70.0,93.620932,193.935589,0.014286,0.261838,1.000000,0.000000,1.252823,11,779.389893,0.0,1.0
20423,69,train_Client_94021,11,107,13/09/2000,1.0,35.0,77.832582,248.761954,0.028571,0.256588,0.971429,0.000000,1.235334,11,1127.753497,0.0,1.0
20424,62,train_Client_76339,11,302,20/11/1984,1.0,34.0,108.056645,58.621138,0.000000,0.241488,1.000000,0.000000,1.041047,11,305.974672,0.0,0.0
20425,60,train_Client_65563,11,101,20/02/2009,1.0,37.0,206.453092,33.241921,0.000000,0.236067,0.972973,0.164399,1.150793,40,254.426190,0.0,0.0


,disrict,client_id,client_catg,region,creation_date,target,1transactions_count,gap_days_nanstd,consumption_per_month_nanstd,frac_time_lvl4_nanmax,consumption_per_month_scaled_nanstd,active_nanmean,counter_statue_nanstd,reading_remarque_nanstd,tarif_type_mode,index_diff_nanstd,has_multi_invoice_same_day_nanmax,used_lvl4_sum
0,62,train_Client_124396,11,309,08/07/2010,0.0,23.0,81.958177,24.251929,0.000000,0.282821,0.391304,0.208514,1.206045,11,154.179840,0.0,0.0
1,60,train_Client_53991,11,101,06/06/1989,0.0,31.0,100.309985,64.004147,0.000000,0.289612,1.000000,0.000000,1.222548,10,270.678494,0.0,0.0
2,62,train_Client_106081,11,303,21/05/2002,0.0,79.0,119.516999,68.579467,0.000000,0.269746,0.987342,0.000000,1.407888,11,313.416317,0.0,0.0
3,62,train_Client_22679,11,310,25/12/2008,0.0,16.0,147.256563,10.416621,0.000000,0.289351,0.937500,0.000000,1.327592,11,45.944895,0.0,0.0
4,62,train_Client_65828,11,303,26/09/2000,0.0,44.0,99.224358,81.875634,0.000000,0.268583,0.931818,0.785707,1.262569,11,392.249837,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13545,63,train_Client_127166,11,306,22/06/2011,1.0,14.0,162.416882,30.368173,0.000000,0.311735,1.000000,0.000000,1.266647,11,358.559511,0.0,0.0
13546,62,train_Client_105991,11,301,21/08/1989,0.0,75.0,80.339197,94.042105,0.000000,0.233679,0.986667,0.115470,1.435960,10,416.131403,0.0,0.0
13547,69,train_Client_109115,51,103,14/04/2009,1.0,135.0,90.018698,407.250064,0.281481,0.210220,0.962963,0.000000,1.269274,11,1109.329157,0.0,38.0
13548,69,train_Client_58137,11,103,11/12/2014,0.0,26.0,82.350611,268.648956,0.038462,0.269220,0.846154,0.000000,1.104536,11,1074.173429,0.0,1.0


,disrict,client_id,client_catg,region,creation_date,1transactions_count,gap_days_nanstd,consumption_per_month_nanstd,frac_time_lvl4_nanmax,consumption_per_month_scaled_nanstd,active_nanmean,counter_statue_nanstd,reading_remarque_nanstd,tarif_type_mode,index_diff_nanstd,has_multi_invoice_same_day_nanmax,used_lvl4_sum
0,62,test_Client_0,11,307,28/05/2002,37.0,44.900834,40.424028,0.000000,0.175757,0.972973,0.000000,1.221061,11,235.684859,0.0,0.0
1,69,test_Client_1,11,103,06/08/2009,22.0,117.157647,722.359649,0.045455,0.194222,1.000000,0.213201,1.216766,11,3200.849986,0.0,1.0
2,62,test_Client_10,11,310,07/04/2004,74.0,13.134901,105.694968,0.013514,0.241883,0.986486,0.000000,1.482216,11,422.779872,0.0,1.0
3,60,test_Client_100,11,101,08/10/1992,40.0,96.299793,66.060236,0.000000,0.293850,0.950000,0.000000,1.034966,11,247.253171,0.0,0.0
4,62,test_Client_1000,11,301,21/07/1977,53.0,113.376261,158.358003,0.000000,0.280807,0.924528,0.686803,1.319443,11,864.294814,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58064,63,test_Client_9995,11,399,17/03/2010,4.0,0.000000,133.188991,0.000000,0.408248,0.500000,0.000000,0.000000,11,532.755964,0.0,0.0
58065,63,test_Client_9996,11,311,28/05/2011,46.0,58.724011,23.653269,0.000000,0.220683,0.978261,0.759163,1.100066,11,107.556838,0.0,0.0
58066,60,test_Client_9997,11,101,04/03/1978,59.0,120.360325,31.634983,0.000000,0.220892,1.000000,0.000000,1.065094,10,116.987086,0.0,0.0
58067,60,test_Client_9998,11,101,23/02/2018,1.0,NaN,NaN,0.000000,NaN,1.000000,NaN,NaN,11,NaN,0.0,0.0


## Tips 
- Thorough EDA and incorporating domain knowledge
- Re-grouping categorical features
- More feature engineering(try utilizing some date-time features)
- Target balancing - oversampling, undersampling, SMOTE, scale_pos_weight
- Model ensembling
- Train-test split or cross-validation


# ******************* GOOD LUCK!!! ***************************